# Prueba de sensibilidad: efecto de `loan_grade` e `loan_int_rate`

**Proyecto Final Integrador – Grupo 10** · Notebook complementario (solo resultados de esta prueba)

**Objetivo.** En el notebook principal se excluyeron `loan_grade` e `loan_int_rate` del modelo, porque podrían generarse después de evaluar al cliente y no estar disponibles al momento real de la predicción. Este notebook mide **cuánto desempeño se pierde** por esa decisión: se entrena el mismo modelo (mismo split, mismo preprocesamiento y mismo procedimiento de umbral) con cuatro conjuntos de variables:

1. Sin ninguna de las dos (modelo adoptado).
2. Agregando solo `loan_int_rate`.
3. Agregando solo `loan_grade`.
4. Agregando ambas.

**Importante:** estos modelos *no* se usan para decidir. Solo sirven para justificar con evidencia la exclusión. Se evalúan los dos algoritmos del proyecto (Regresión Logística y Random Forest).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, roc_auc_score, average_precision_score,
                             precision_recall_curve, recall_score, precision_score, f1_score)

pd.set_option('display.max_columns', None)

# Carga del dataset desde Drive
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/credit_risk_dataset.csv")

## 1. Datos y limpieza (idénticos al notebook principal)

In [ ]:
df_limpio = df.copy()
df_limpio = df_limpio[~df_limpio['person_age'].isin([123, 144])]
df_limpio = df_limpio[df_limpio['person_emp_length'].isna() |
                      (df_limpio['person_emp_length'] <= (df_limpio['person_age'] - 14))]
df_limpio = df_limpio.drop_duplicates()

print("Filas tras la limpieza:", len(df_limpio))   # debe coincidir con el notebook principal: 32.409

## 2. División entrenamiento/test y preprocesamiento

Misma división 80/20 estratificada (`random_state=42`) que en el notebook principal. Se conservan todas las columnas para poder agregar luego `loan_grade` y `loan_int_rate`; cada variante selecciona las columnas que usa. Los nulos de `person_emp_length` y de `loan_int_rate` se imputan con la mediana del entrenamiento.

In [ ]:
X_completo = df_limpio.drop(columns=['loan_status'])
y = df_limpio['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X_completo, y, test_size=0.20, random_state=42, stratify=y)

print("Entrenamiento:", X_train.shape, "| Test:", X_test.shape)

variables_numericas = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
                       'loan_percent_income', 'cb_person_cred_hist_length']
variables_categoricas = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']

pipeline_numerico = Pipeline(steps=[('imputacion', SimpleImputer(strategy='median')),
                                    ('escalado', StandardScaler())])
pipeline_categorico = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 3. Funciones de entrenamiento y evaluación

El umbral se elige igual que en el notebook principal: con probabilidades obtenidas por validación cruzada estratificada (5 particiones) **sobre el entrenamiento**, se toma el umbral de mayor Precision entre los que alcanzan un Recall ≥ 85% en la clase mora. Recién después se evalúa en test.

In [ ]:
def armar_modelo(algoritmo, num_extra, cat_extra):
    preprocesador = ColumnTransformer(transformers=[
        ('num', pipeline_numerico, variables_numericas + num_extra),
        ('cat', pipeline_categorico, variables_categoricas + cat_extra)])
    if algoritmo == 'Regresión Logística':
        estimador = LogisticRegression(max_iter=1000, random_state=42)
    else:
        estimador = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    return Pipeline(steps=[('preprocesamiento', preprocesador), ('modelo', estimador)])

def umbral_recall_minimo(modelo, recall_objetivo=0.85):
    prob_cv = cross_val_predict(modelo, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
    prec, rec, thr = precision_recall_curve(y_train, prob_cv)
    idx = np.argmax(np.where(rec[:-1] >= recall_objetivo, prec[:-1], -1))
    return thr[idx]

def evaluar(algoritmo, variante, num_extra, cat_extra):
    modelo = armar_modelo(algoritmo, num_extra, cat_extra)
    modelo.fit(X_train, y_train)
    prob = modelo.predict_proba(X_test)[:, 1]
    umbral = umbral_recall_minimo(modelo)
    pred = (prob >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {'Algoritmo': algoritmo, 'Variables': variante, 'Umbral': round(umbral, 3),
            'Recall mora': round(recall_score(y_test, pred), 3),
            'Precision mora': round(precision_score(y_test, pred), 3),
            'F1 mora': round(f1_score(y_test, pred), 3),
            'ROC-AUC': round(roc_auc_score(y_test, prob), 3),
            'PR-AUC': round(average_precision_score(y_test, prob), 3),
            'Falsos positivos': fp, 'Falsos negativos': fn}

## 4. Ejecución de las cuatro variantes

In [ ]:
variantes = {'1. Sin grade ni tasa (adoptado)': ([], []),
             '2. Con loan_int_rate': (['loan_int_rate'], []),
             '3. Con loan_grade': ([], ['loan_grade']),
             '4. Con loan_grade y loan_int_rate': (['loan_int_rate'], ['loan_grade'])}

resultados = []
for algoritmo in ['Regresión Logística', 'Random Forest']:
    for nombre, (n_extra, c_extra) in variantes.items():
        resultados.append(evaluar(algoritmo, nombre, n_extra, c_extra))

tabla_sensibilidad = pd.DataFrame(resultados)
tabla_sensibilidad

## 5. Resumen para el informe

In [ ]:
resumen = tabla_sensibilidad.pivot(index='Variables', columns='Algoritmo', values=['ROC-AUC', 'PR-AUC', 'Precision mora', 'Falsos positivos'])
resumen

## 6. Conclusión de la prueba

Con el mismo procedimiento y un Recall objetivo de ≈ 85%, agregar `loan_grade` y/o `loan_int_rate` mejora el desempeñoo.

La mejora es mayor en Random Forest (ROC-AUC de 0,877 a 0,936; falsos positivos casi a la mitad). Esto confirma que ambas variables contienen mucha información sobre el riesgo, pero justamente porque son el resultado de una evaluación crediticia previa (el grado y la tasa asignados ya reflejan la opinión de la entidad sobre el riesgo del cliente). Si la predicción debe hacerse antes de esa asignación, esas variables no están disponibles y un modelo que las use no sería aplicable en la práctica.

Por lo tanto, **el modelo adoptado (sin ambas variables) representa el desempeño alcanzable al momento de la solicitud, y el modelo con ambas debe leerse como una cota superior de referencia**, no como un resultado utilizable. Si la entidad dispusiera de grado y tasa antes de decidir, podría reevaluarse su inclusión.